# AC-MOT v16 — Final Paper Evaluation
This notebook is evaluation-only. It preserves the frozen v15 proposed system and compares quality + realtime processing against fixed baselines and ablations. Each stage explains what is running and why.


In [ ]:
import time
NOTEBOOK_START=time.perf_counter(); STAGE_TIMES={}
def fmt_time(s):
    s=max(float(s),0); m,sec=divmod(int(round(s)),60)
    return f'{s:.1f}s' if s<60 else f'{m}m {sec:02d}s'
def begin(n,title,now,why,remaining):
    print('\n'+'='*100,flush=True); print(f'[{n}/6] {title}',flush=True)
    print(f'[NOW] {now}',flush=True); print(f'[WHY] {why}',flush=True)
    print(f'[REMAINING AFTER THIS] {remaining}',flush=True); return time.perf_counter()
def end(n,t):
    e=time.perf_counter()-t; STAGE_TIMES[n]=e
    print(f'[DONE {n}/6] elapsed={fmt_time(e)} | total={fmt_time(time.perf_counter()-NOTEBOOK_START)}',flush=True)
t=begin(1,'Mount Google Drive','Mounting the current account Drive.','v16 auto-discovers VisDrone and writes persistent paper results.','GitHub -> dependencies -> config -> preflight -> paper evaluation')
from google.colab import drive
drive.mount('/content/drive')
end(1,t)


## Stage 2 — Authenticate and sync GitHub
Requires a Colab Secret named `GITHUB_TOKEN` from a GitHub account authorized to read the private `AhmedCode110/AC-MOT` repository. The token is never printed or embedded in the clone URL.


In [ ]:
t=begin(2,'Sync private AC-MOT repository','Validating GITHUB_TOKEN and pulling main.','Ensures Colab executes the exact latest v16 source of truth.','dependencies -> config -> preflight -> paper evaluation')
from pathlib import Path
import json, os, sys, subprocess, tempfile, urllib.request, urllib.error
from google.colab import userdata
REPO=Path('/content/AC-MOT'); REPO_URL='https://github.com/AhmedCode110/AC-MOT.git'; USER_API='https://api.github.com/user'; REPO_API='https://api.github.com/repos/AhmedCode110/AC-MOT'
token=userdata.get('GITHUB_TOKEN')
if not token: raise RuntimeError('Missing Colab Secret GITHUB_TOKEN or Notebook access is disabled.')
def gh(url):
    req=urllib.request.Request(url,headers={'Authorization':f'Bearer {token}','Accept':'application/vnd.github+json','User-Agent':'AC-MOT-v16'})
    with urllib.request.urlopen(req,timeout=20) as r: return r.status,json.loads(r.read().decode())
status,user=gh(USER_API)
if status!=200: raise RuntimeError('GITHUB_TOKEN authentication failed')
status,repo=gh(REPO_API)
if status!=200: raise RuntimeError('Token cannot read AhmedCode110/AC-MOT')
print(f'[OK] GitHub user={user.get("login")} | repo={repo.get("full_name")}',flush=True)
with tempfile.TemporaryDirectory() as tmp:
    env=os.environ.copy(); env['GIT_TERMINAL_PROMPT']='0'; env['ACMOT_GH_TOKEN']=token; env['ACMOT_GH_USER']=user.get('login','x')
    helper=Path(tmp)/'askpass'; helper.write_text('#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ""\nprint(os.environ["ACMOT_GH_USER"] if "Username" in p else os.environ["ACMOT_GH_TOKEN"])\n'); helper.chmod(0o700); env['GIT_ASKPASS']=str(helper)
    base=['git','-c','credential.helper=']
    cmd=base+(['-C',str(REPO),'pull','--ff-only','origin','main'] if (REPO/'.git').is_dir() else ['clone','--branch','main',REPO_URL,str(REPO)])
    r=subprocess.run(cmd,env=env,text=True,capture_output=True); print(r.stdout,flush=True); print(r.stderr,flush=True); r.check_returncode()
commit=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
print(f'[OK] repo commit={commit}',flush=True)
end(2,t)


## Stage 3 — Install pinned dependencies
Installs the repository requirements and pinned TrackEval revision. Network/package work has no trustworthy ETA, so only elapsed time is reported.


In [ ]:
t=begin(3,'Install pinned environment','Installing requirements and TrackEval.','Prevents package drift from changing FPS or metrics.','config -> preflight -> paper evaluation')
subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO/'requirements.txt')],check=True)
subprocess.run([sys.executable,str(REPO/'scripts/setup_trackeval.py'),'/content/TrackEval'],check=True)
end(3,t)


## Stage 4 — Resolve v16 portable configuration
Loads `paper_eval_v16.json`, automatically locates the verified 17-sequence VisDrone split, and chooses the current account's writable v16 results folder.


In [ ]:
t=begin(4,'Resolve v16 config and Drive paths','Loading active config and auto-finding dataset/output.','Makes the paper evaluation portable across authorized Colab accounts.','preflight -> paper evaluation')
import uuid
CONFIG_NAME=(REPO/'configs'/'active_config.txt').read_text().strip(); CFG=json.loads((REPO/'configs'/CONFIG_NAME).read_text())
if CFG.get('version')!='v16' or CFG.get('mode')!='paper_eval': raise RuntimeError(f'Expected active v16 paper_eval, got {CFG.get("version")} / {CFG.get("mode")}')
if str(REPO) not in sys.path: sys.path.insert(0,str(REPO))
from portable_v16 import resolve_portable_config, portable_requirements_text
print('[PORTABLE] '+portable_requirements_text(),flush=True)
CFG=resolve_portable_config(CFG,verbose=True)
CONFIG_PATH=Path('/content')/('acmot_v16_'+uuid.uuid4().hex+'.json'); CONFIG_PATH.write_text(json.dumps(CFG,indent=2))
print(f'[CONFIG] systems={len(CFG["systems"])} | target={CFG["target_fps"]} FPS',flush=True)
print(f'[CONFIG] dataset={CFG["dataset"]}',flush=True); print(f'[CONFIG] output={CFG["output_root"]}',flush=True)
end(4,t)


## Stage 5 — Preflight
Checks CUDA/Tesla T4, package pins, dataset resolution and v16 paper-evaluation routing before the expensive full comparison.


In [ ]:
t=begin(5,'Environment preflight','Checking T4, packages, v16 mode and resolved paths.','Stops early if the experiment cannot be reproduced correctly.','paper evaluation')
subprocess.run([sys.executable,str(REPO/'scripts/run.py'),'--config',str(CONFIG_PATH),'--check'],check=True)
end(5,t)


## Stage 6 — Full paper evaluation
Runs 9 systems over all 17 sequences / 6635 frames. Progress lines show system, sequence, frame, processing FPS, resolution and ETA. After tracking, pinned TrackEval calculates HOTA/DetA/AssA/MOTA/IDF1/IDS/FN/FP and v16 writes `paper_comparison.csv` and `paper_comparison.md`.


In [ ]:
t=begin(6,'Run v16 final paper evaluation','Running full tracking, timing, TrackEval and paper-table generation.','This is evaluation only; do not retune v15 from these results.','none — final stage')
print(f'[RUN] version=v16 | systems={len(CFG["systems"])} | dataset_frames=6635 | target={CFG["target_fps"]} FPS',flush=True)
subprocess.run([sys.executable,str(REPO/'scripts/run.py'),'--config',str(CONFIG_PATH)],check=True)
end(6,t)
print('\n'+'='*100,flush=True); print('[OK] V16 PAPER EVALUATION COMPLETED',flush=True)
print(f'[TOTAL] {fmt_time(time.perf_counter()-NOTEBOOK_START)}',flush=True); print(f'[RESULTS ROOT] {CFG["output_root"]}',flush=True)
print('[STAGE TIMES]',{k:fmt_time(v) for k,v in STAGE_TIMES.items()},flush=True)
